# Cubes and Collections

For multi-dimensional data (e.g., scenario x time, or region x time) use `TimeSeriesCube`. For grouping heterogeneous time series that don't share the same timestamps, use `TimeSeriesCollection`.

## TimeSeriesCube

A cube stores an N-dimensional array with named `Dimension` objects. Common use cases include ensemble forecasts, scenario analysis, and region-by-time grids.

In [1]:
from datetime import datetime, timedelta, timezone

import numpy as np

import timedatamodel as tdm

base = datetime(2024, 1, 15, tzinfo=timezone.utc)
hours = [base + timedelta(hours=i) for i in range(24)]

ModuleNotFoundError: No module named 'timedatamodel'

### Building a cube from scratch

Create a 3-scenario x 24-hour cube representing price forecasts under different assumptions.

In [2]:
rng = np.random.default_rng(42)
base_prices = 50 + 20 * np.sin(np.linspace(0, 2 * np.pi, 24))

data = np.array([
    base_prices * 0.8 + rng.normal(0, 2, 24),  # low scenario
    base_prices + rng.normal(0, 2, 24),          # base scenario
    base_prices * 1.3 + rng.normal(0, 3, 24),   # high scenario
])

cube = tdm.TimeSeriesCube(
    tdm.Frequency.PT1H,
    timezone="UTC",
    name="price_forecast",
    unit="EUR/MWh",
    data_type=tdm.DataType.FORECAST,
    dimensions=[
        tdm.Dimension("scenario", ["low", "base", "high"]),
        tdm.Dimension("valid_time", hours),
    ],
    values=data,
)
cube

NameError: name 'tdm' is not defined

### Cube properties

In [3]:
print(f"Shape:      {cube.shape}")
print(f"Dimensions: {cube.dim_names}")
print(f"Begin:      {cube.begin}")
print(f"End:        {cube.end}")
print(f"Has missing:{cube.has_missing}")

NameError: name 'cube' is not defined

### Selecting with `sel()` — label-based

Select a single scenario to collapse the cube into a `TimeSeries`.

In [4]:
base_scenario = cube.sel(scenario="base")
base_scenario

NameError: name 'cube' is not defined

### Selecting with `isel()` — index-based

Select by integer position.

In [5]:
first_scenario = cube.isel(scenario=0)
first_scenario

NameError: name 'cube' is not defined

### Slicing a dimension

Select a range of labels to get a smaller cube or table.

In [6]:
two_scenarios = cube.sel(scenario=slice("low", "base"))
print(f"Type:  {type(two_scenarios).__name__}")
print(two_scenarios)

NameError: name 'cube' is not defined

### Auto-collapse to Table or Series

When a `sel()` or `isel()` call removes enough dimensions, the result automatically becomes a `TimeSeriesTable` (2D) or `TimeSeries` (1D).

In [7]:
table_view = cube.to_table()
print(f"Type: {type(table_view).__name__}")
table_view

NameError: name 'cube' is not defined

### Building a cube from a list of TimeSeries

`from_timeseries_list()` is handy when you already have individual scenario forecasts.

In [8]:
series_list = [
    tdm.TimeSeries(
        tdm.Frequency.PT1H,
        timestamps=hours,
        values=(base_prices * factor + rng.normal(0, 2, 24)).tolist(),
        name="price",
        unit="EUR/MWh",
    )
    for factor in [0.7, 0.85, 1.0, 1.15, 1.3]
]

ensemble = tdm.TimeSeriesCube.from_timeseries_list(
    series_list,
    dimension=tdm.Dimension("percentile", ["p10", "p25", "p50", "p75", "p90"]),
    name="price_ensemble",
)
ensemble

NameError: name 'tdm' is not defined

---

## TimeSeriesCollection

A `TimeSeriesCollection` groups time series that may have different frequencies, time ranges, or numbers of points. Think of it as a named bag of `TimeSeries` and `TimeSeriesTable` objects.

In [9]:
daily_base = datetime(2024, 1, 1, tzinfo=timezone.utc)

ts_hourly = tdm.TimeSeries(
    tdm.Frequency.PT1H,
    timestamps=hours,
    values=[100.0 + rng.normal(0, 10) for _ in range(24)],
    name="wind_hourly",
    unit="MW",
)

ts_daily = tdm.TimeSeries(
    tdm.Frequency.P1D,
    timestamps=[daily_base + timedelta(days=d) for d in range(30)],
    values=[2400.0 + rng.normal(0, 200) for _ in range(30)],
    name="wind_daily_energy",
    unit="MWh",
)

ts_15min = tdm.TimeSeries(
    tdm.Frequency.PT15M,
    timestamps=[base + timedelta(minutes=15 * i) for i in range(96)],
    values=[50.0 + rng.normal(0, 5) for _ in range(96)],
    name="solar_15min",
    unit="MW",
)

NameError: name 'tdm' is not defined

### Creating a collection

In [10]:
collection = tdm.TimeSeriesCollection(
    [ts_hourly, ts_daily, ts_15min],
    name="Plant overview",
    description="Mixed-frequency data for a single plant",
)
collection

NameError: name 'tdm' is not defined

### Dictionary-like access

In [11]:
print(f"Names: {collection.names}")
print(f"Count: {collection.series_count}")

collection["wind_hourly"]

NameError: name 'collection' is not defined

### Adding and removing series

Collections are immutable — `add()` and `remove()` return new collections.

In [12]:
ts_price = tdm.TimeSeries(
    tdm.Frequency.PT1H,
    timestamps=hours,
    values=[45.0 + rng.normal(0, 8) for _ in range(24)],
    name="spot_price",
    unit="EUR/MWh",
)

extended = collection.add(ts_price)
print(f"Original: {collection.names}")
print(f"Extended: {extended.names}")

NameError: name 'tdm' is not defined

In [13]:
reduced = extended.remove("wind_daily_energy")
print(f"Reduced: {reduced.names}")

NameError: name 'extended' is not defined

### Iterating over a collection

In [14]:
for name, series in collection.items():
    print(f"{name:20s}  freq={str(series.frequency):5s}  len={len(series):3d}  begin={series.begin}")

NameError: name 'collection' is not defined

## Summary

- **`TimeSeriesCube`**: N-dimensional time series with `Dimension` labels; slice with `sel()` / `isel()`; auto-collapses to Table or Series
- **`TimeSeriesCollection`**: heterogeneous container for series with different frequencies and time ranges; dictionary-like access; immutable add/remove

Next up: **nb_07** covers data quality tools — coverage bars and validation.